In [ ]:
%cd ..

In [ ]:
ENTITY = input("Your wandb entity")
SWEEP_ID = input("The wandb sweep ID of the classification experiment")

import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from tqdm.notebook import tqdm
import json
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial

def fetch_run_data(run):
    """Fetch data for a single run - to be run in parallel."""
    try:
        stats = dict(run.config)
        stats.update(run.summary)
        stats["run_id"] = run.id
        stats["state"] = run.state
        return stats
    except Exception as e:
        print(f"Error fetching run {run.id}: {e}")
        return None

def load_sweep(sweep_id, max_runs=1000000000, max_workers=10, batch_size=100):
    """
    Load sweep runs in parallel for faster processing.
    
    Args:
        sweep_id: W&B sweep ID
        max_runs: Maximum number of runs to fetch
        max_workers: Number of parallel workers (threads) for fetching run details
        batch_size: Number of runs to fetch before processing in parallel
    """
    api = wandb.Api(timeout=100)

    warnings.filterwarnings("ignore", message="DataFrame is highly fragmented")

    # Collect configs and full history for each run
    all_runs_data = []

    # Use api.runs() with filters to stream runs
    runs = api.runs(
        path=f"{ENTITY}/softmax",
        filters={"sweep": sweep_id},
        per_page=50
    )

    # Convert to list iterator so we can batch it
    runs_iter = iter(runs)
    total_processed = 0
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        while total_processed < max_runs:
            # Get a batch of runs
            batch = []
            for _ in range(batch_size):
                try:
                    run = next(runs_iter)
                    batch.append(run)
                    total_processed += 1
                    if total_processed >= max_runs:
                        break
                except StopIteration:
                    break
            
            if not batch:
                break
            
            # Process batch in parallel
            futures = {executor.submit(fetch_run_data, run): run for run in batch}
            
            # Collect results with progress bar
            for future in tqdm(as_completed(futures), total=len(futures), 
                             desc=f"Processing runs {total_processed-len(batch)+1}-{total_processed}"):
                result = future.result()
                if result is not None:
                    all_runs_data.append(result)

    # Concatenate all runs into one long DataFrame
    df = pd.DataFrame(all_runs_data)

    # Display info about the DataFrame
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    return df

In [ ]:
data = load_sweep(SWEEP_ID)
data.to_csv("data/classification_data.csv")

# MAIN

In [ ]:
import re
from itertools import product
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
data = pd.read_csv("data/classification_data.csv")
data = data[data["attention_type"] != "uniform"]
data["attention_type"] = data["attention_type"].apply(lambda x: "linear_norm" if x == "linear" else x)

# ICML-style settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'hatch.linewidth': 0.1,
})

NORMALIZED = "w. norm."
UNNORMALIZED = "w/o norm."

def extract_base_and_norm(attn_type):
    if attn_type == 'softmax':
        return 'softmax', NORMALIZED
    elif attn_type.endswith('_norm'):
        return attn_type[:-5], NORMALIZED
    elif attn_type.endswith('_unnorm'):
        return attn_type[:-7], UNNORMALIZED
    else:
        return attn_type, None

rename_map = {
    'linear': r'elu $\cdot$ elu',
    'coupled_linear': 'elu',
    'raw_dotproduct': 'linear',
    'softmax': 'softmax',
    'sigmoid': 'sigmoid',
    'mlp': 'mlp',
}

hls = sns.color_palette("hls", 8)
colors = [hls[3], hls[5]]
desired_order = ['softmax', 'sigmoid', 'elu', r'elu $\cdot$ elu', 'mlp', 'linear']

# Base filter conditions
base_filter = data[data['vocab_size'] == 41]

# Get unique values of num_layers and num_heads
num_layers_vals = sorted(base_filter['num_layers'].unique())
num_heads_vals = sorted(base_filter['num_heads'].unique())

# Find all columns matching pattern <i>_<j>_rel_attn_max
pattern = re.compile(r'^\d+_\d+_rel_attn_max$')
rel_attn_cols = [col for col in data.columns if pattern.match(col)]
print(f"Found rel_attn_max columns: {rel_attn_cols}")

# All combinations of use_mlp and use_layernorm
mlp_ln_combos = [(False, False), (False, True), (True, False), (True, True)]

for n_layers, n_heads in product(num_layers_vals, num_heads_vals):
    fig, axes = plt.subplots(1, 4, figsize=(13, 1.8))
    
    for ax_idx, (use_mlp, use_ln) in enumerate(mlp_ln_combos):
        ax = axes[ax_idx]
        
        subset = base_filter[
            (base_filter['num_layers'] == n_layers) &
            (base_filter['num_heads'] == n_heads) &
            (base_filter['use_mlp'] == use_mlp) &
            (base_filter['use_layernorm'] == use_ln)
        ].copy()
        
        if len(subset) == 0:
            ax.set_visible(False)
            continue
        
        # Compute average of all rel_attn_max columns (ignoring NaN)
        subset['dominance_rate'] = subset[rel_attn_cols].clip(0, 1).mean(axis=1, skipna=True)
        
        # Extract base attention type and norm status
        subset['base_attention_type'] = subset['attention_type'].apply(lambda x: extract_base_and_norm(x)[0])
        subset['norm_type'] = subset['attention_type'].apply(lambda x: extract_base_and_norm(x)[1])
        subset['base_attention_type'] = subset['base_attention_type'].map(lambda x: rename_map.get(x, x))
        subset = subset[subset['norm_type'].notna()]
        
        # Always use full order
        order = desired_order
        
        # Track which combinations have data
        existing_combinations = set()
        for _, row in subset.iterrows():
            existing_combinations.add((row['base_attention_type'], row['norm_type']))
        
        # Plot bars if there's data
        if len(subset) > 0:
            sns.barplot(
                data=subset,
                x='base_attention_type',
                y='dominance_rate',
                hue='norm_type',
                hue_order=[NORMALIZED, UNNORMALIZED],
                order=order,
                palette=colors,
                ax=ax,
                errorbar=None,
                edgecolor='black',
                linewidth=0.5,
                width=0.7,
                gap=0.1,
            )
            
            # Add subtle hatching to all bars
            for patch in ax.patches:
                patch.set_hatch('*********')
        
        # Calculate bar positions and add red crosses for missing data
        n_hues = 2
        width = 0.7
        gap = 0.1
        bar_width = width / n_hues - gap / n_hues
        
        for i, attn_type in enumerate(order):
            for j, norm_type in enumerate([NORMALIZED, UNNORMALIZED]):
                # Skip softmax unnormalized (doesn't exist by design)
                if attn_type == 'softmax' and norm_type == UNNORMALIZED:
                    continue
                
                if (attn_type, norm_type) not in existing_combinations:
                    # Calculate x position for the missing bar
                    x_pos = i + (j - 0.5) * (bar_width + gap / 2)
                    # Add red cross
                    ax.plot(x_pos, 0.1, 'x', color='red', markersize=7, markeredgewidth=2)
        
        # Shift softmax bar to center if it exists
        softmax_idx = order.index('softmax')
        if ('softmax', NORMALIZED) in existing_combinations:
            for idx, patch in enumerate(ax.patches):
                if idx == softmax_idx:
                    x = patch.get_x()
                    w = patch.get_width()
                    patch.set_x(x + w / 1.7)
        
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels(order)
        ax.set_xlabel('')
        ax.set_ylim(0, 1)
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
        ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
        ax.tick_params(axis='x', rotation=45)
        
        # Only first subplot gets y-label
        if ax_idx == 0:
            ax.set_ylabel('Sparsity Score')
        else:
            ax.set_ylabel('')
        
        # Subplot title
        mlp_str = "MLP" if use_mlp else "no MLP"
        ln_str = "LN" if use_ln else "no LN"
        ax.set_title(f"{mlp_str}, {ln_str}")
        
        # Remove legend from individual subplots
        if ax.get_legend():
            ax.get_legend().remove()
    
    # Add shared legend
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper right', frameon=False, ncol=2)
    
    fig.suptitle(f"layers={int(n_layers)}, heads={int(n_heads)}", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
import re
import os
from itertools import product
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
data = pd.read_csv("data/classification_data.csv")
data = data[data["attention_type"] != "uniform"]
data["attention_type"] = data["attention_type"].apply(lambda x: "linear_norm" if x == "linear" else x)

# ICML-style settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'hatch.linewidth': 0.1,
})

NORMALIZED = "w. norm."
UNNORMALIZED = "w/o norm."

def extract_base_and_norm(attn_type):
    if attn_type == 'softmax':
        return 'softmax', NORMALIZED
    elif attn_type.endswith('_norm'):
        return attn_type[:-5], NORMALIZED
    elif attn_type.endswith('_unnorm'):
        return attn_type[:-7], UNNORMALIZED
    else:
        return attn_type, None

rename_map = {
    'linear': r'elu $\cdot$ elu',
    'coupled_linear': 'elu',
    'raw_dotproduct': 'linear',
    'softmax': 'softmax',
    'sigmoid': 'sigmoid',
    'mlp': 'mlp',
}

hls = sns.color_palette("hls", 8)
colors = [hls[3], hls[5]]
desired_order = ['softmax', 'sigmoid', 'elu', r'elu $\cdot$ elu', 'mlp', 'linear']

# Filter for 1 layer and 1 head
base_filter = data[(data['num_layers'] == 1) & (data['num_heads'] == 1)]

# Get unique vocab sizes
vocab_sizes = sorted(base_filter['vocab_size'].unique())

# Find all columns matching pattern mixing/<i>_<j>/avg_top1_dominance_rate
pattern = re.compile(r'^mixing/\d+_\d+/avg_top1_dominance_rate$')
dominance_cols = [col for col in data.columns if pattern.match(col)]
print(f"Found dominance columns: {dominance_cols}")

output_dir = 'visualizations/fig'

for vocab_size, use_mlp, use_ln in product(vocab_sizes, [False, True], [False, True]):
    subset = base_filter[
        (base_filter['vocab_size'] == vocab_size) &
        (base_filter['use_mlp'] == use_mlp) &
        (base_filter['use_layernorm'] == use_ln)
    ].copy()
    
    if len(subset) == 0:
        continue
    
    # Compute max of all dominance columns (ignoring NaN)
    subset['dominance_rate'] = subset[dominance_cols].max(axis=1, skipna=True)
    
    # Extract base attention type and norm status
    subset['base_attention_type'] = subset['attention_type'].apply(lambda x: extract_base_and_norm(x)[0])
    subset['norm_type'] = subset['attention_type'].apply(lambda x: extract_base_and_norm(x)[1])
    subset['base_attention_type'] = subset['base_attention_type'].map(lambda x: rename_map.get(x, x))
    subset = subset[subset['norm_type'].notna()]
    
    fig, ax = plt.subplots(figsize=(3.25, 1.8))
    
    # Always use full order
    order = desired_order
    
    # Track which combinations have data
    existing_combinations = set()
    for _, row in subset.iterrows():
        existing_combinations.add((row['base_attention_type'], row['norm_type']))
    
    # Plot bars if there's data
    if len(subset) > 0:
        sns.barplot(
            data=subset,
            x='base_attention_type',
            y='dominance_rate',
            hue='norm_type',
            hue_order=[NORMALIZED, UNNORMALIZED],
            order=order,
            palette=colors,
            ax=ax,
            errorbar=None,
            edgecolor='black',
            linewidth=0.5,
            width=0.7,
            gap=0.1,
        )
        
        # Add subtle hatching to all bars
        for patch in ax.patches:
            patch.set_hatch('....')
    
    # Calculate bar positions and add red crosses for missing data
    n_hues = 2
    width = 0.7
    gap = 0.1
    bar_width = width / n_hues - gap / n_hues
    
    for i, attn_type in enumerate(order):
        for j, norm_type in enumerate([NORMALIZED, UNNORMALIZED]):
            # Skip softmax unnormalized (doesn't exist by design)
            if attn_type == 'softmax' and norm_type == UNNORMALIZED:
                continue
            
            if (attn_type, norm_type) not in existing_combinations:
                # Calculate x position for the missing bar
                x_pos = i + (j - 0.5) * (bar_width + gap / 2)
                # Add red cross
                ax.plot(x_pos, 0.1, 'x', color='red', markersize=7, markeredgewidth=2)
    
    # Shift softmax bar to center if it exists
    softmax_idx = order.index('softmax')
    if ('softmax', NORMALIZED) in existing_combinations:
        for idx, patch in enumerate(ax.patches):
            if idx == softmax_idx:
                x = patch.get_x()
                w = patch.get_width()
                patch.set_x(x + w / 1.7)
    
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order)
    ax.set_xlabel('')
    ax.set_ylabel('Flip Rate')
    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
    ax.tick_params(axis='x', rotation=45)
    
    # Add legend
    if len(subset) > 0:
        ax.legend(title='', frameon=False, loc='upper right', ncol=2)
    
    plt.tight_layout()
    
    # Save with descriptive filename
    mlp_str = "mlp" if use_mlp else "nomlp"
    ln_str = "ln" if use_ln else "noln"
    filename = f"fliprate_vocab{int(vocab_size)}_{mlp_str}_{ln_str}.pdf"
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()
    print(f"Saved: {filename}")

print("Done!")